In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import copy


In [2]:
print("Loading the Parquet masterpieces...")
train_df = pd.read_parquet("../data/model_ready/train_final.parquet")
valid_df = pd.read_parquet("../data/model_ready/valid_final.parquet")
test_df = pd.read_parquet("../data/model_ready/test_final.parquet")

target_col = 'SalaryNormalized'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# THE KAGGLE CHEAT CODE (Log Transform)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)
y_test_log = np.log1p(y_test)

Loading the Parquet masterpieces...


In [3]:
#==========================================
# 1. SCALE THE DATA (Neural Nets demand it)
# ==========================================
print("Scaling the data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)


Scaling the data...


In [4]:
# ==========================================
# 2. CONVERT TO PYTORCH TENSORS
# ==========================================
print("Moving data to PyTorch format...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert to float32 tensors and push to GPU
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train_log.values, dtype=torch.float32).view(-1, 1).to(device)

X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32).to(device)
y_valid_t = torch.tensor(y_valid_log.values, dtype=torch.float32).view(-1, 1).to(device)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

# Create DataLoaders (Batches of 128 so your 6GB VRAM doesn't cry)
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)


Moving data to PyTorch format...


In [5]:
# ==========================================
# 3. BUILD THE ARCHITECTURE
# ==========================================
class SalaryPredictorMLP(nn.Module):
    def __init__(self, input_dim):
        super(SalaryPredictorMLP, self).__init__()
        
        # Layer 1: The massive funnel (Input -> 512)
        self.layer1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3) # Turns off 30% of neurons randomly
        
        # Layer 2: Pattern recognition (512 -> 256)
        self.layer2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
        
        # Layer 3: Squeezing the vibes (256 -> 64)
        self.layer3 = nn.Linear(256, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu3 = nn.ReLU()
        self.drop3 = nn.Dropout(0.2)
        
        # Output Layer: One single continuous number (The log salary)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = self.drop1(self.relu1(self.bn1(self.layer1(x))))
        x = self.drop2(self.relu2(self.bn2(self.layer2(x))))
        x = self.drop3(self.relu3(self.bn3(self.layer3(x))))
        return self.out(x)

# Initialize the model, loss function, and optimizer
input_size = X_train_t.shape[1]
model = SalaryPredictorMLP(input_size).to(device)

# L1 Loss (MAE) is usually better for salaries, but MSE keeps gradients smooth. We'll use MSE for training.
criterion = nn.MSELoss() 
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)


In [6]:

# ==========================================
# 4. THE TRAINING LOOP WITH EARLY STOPPING
# ==========================================
epochs = 100
patience = 10  # Stop if valid loss doesn't improve for 10 epochs
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = None

print("Training the Deep Learning Brain...")

for epoch in range(epochs):
    model.train() # Tell PyTorch we are learning
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Clear old gradients
        predictions = model(batch_X)    # Guess the salary
        loss = criterion(predictions, batch_y) # Calculate how wrong it is
        loss.backward()                 # Backpropagation (Learn)
        optimizer.step()                # Update weights
        train_loss += loss.item() * batch_X.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # --- Validation Phase ---
    model.eval() # Tell PyTorch to freeze (turns off Dropout & BatchNorm tracking)
    with torch.no_grad():
        val_preds = model(X_valid_t)
        val_loss = criterion(val_preds, y_valid_t).item()
        
    # --- Early Stopping Logic ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_weights = copy.deepcopy(model.state_dict()) # Save the winning brain
    else:
        patience_counter += 1
        
    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Valid Loss: {val_loss:.4f} | Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"🛑 Early Stopping triggered at Epoch {epoch}. Loading best weights.")
        break

# Load the best weights we saved
model.load_state_dict(best_model_weights)


Training the Deep Learning Brain...
Epoch 00 | Train Loss: 6.8324 | Valid Loss: 0.0970 | Patience: 0/10
Epoch 05 | Train Loss: 0.4387 | Valid Loss: 0.0559 | Patience: 0/10
Epoch 10 | Train Loss: 0.1988 | Valid Loss: 0.0703 | Patience: 1/10
Epoch 15 | Train Loss: 0.0945 | Valid Loss: 0.0567 | Patience: 6/10
🛑 Early Stopping triggered at Epoch 19. Loading best weights.


<All keys matched successfully>

In [7]:

# ==========================================
# 5. FINAL PREDICTIONS & FLEXING
# ==========================================
print("\nPredicting on Test Set...")
model.eval()
with torch.no_grad():
    test_preds_log = model(X_test_t).cpu().numpy().flatten()

# Reverse the log cheat code
test_preds_real = np.expm1(test_preds_log)

# Metrics
mae = mean_absolute_error(y_test, test_preds_real)
r2 = r2_score(y_test, test_preds_real)

print("\n" + "="*30)
print("🤖 DEEP NEURAL NETWORK RESULTS 🤖")
print("="*30)
print(f"R2 Score: {r2:.4f}")
print(f"MAE:      £{mae:.2f}")
print("="*30 + "\n")


Predicting on Test Set...

🤖 DEEP NEURAL NETWORK RESULTS 🤖
R2 Score: 0.5906
MAE:      £5611.60

